In [15]:
from google.colab import drive
drive.mount ("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
os.chdir('/content/drive/MyDrive/Data-Analytics-Projects')

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ulrikthygepedersen/online-retail-dataset")

print("Path to dataset files:", path)

100%|██████████| 7.38M/7.38M [00:00<00:00, 180MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ulrikthygepedersen/online-retail-dataset/versions/2


In [11]:
!unzip -q archive.zip

In [13]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Data-Analytics-Projects/online_retail.csv')

df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [14]:
import pandas as pd

def load_and_clean_data(file_path: str) -> pd.DataFrame:
    """
    Loads and cleans the Online Retail dataset for customer segmentation.
    """
    print("Loading dataset...")
    # Adjust encoding if necessary (latin1 is standard for this UCI dataset)
    df = pd.read_csv(file_path, encoding='latin1')

    initial_rows = len(df)
    print(f"Initial row count: {initial_rows:,}")

    # 1. Drop rows missing CustomerID
    df = df.dropna(subset=['CustomerID'])

    # 2. Filter out cancelled orders
    # Cancelled orders usually have Quantity < 0 or InvoiceNo starting with 'C'
    df = df[df['Quantity'] > 0]

    # Convert InvoiceNo to string to safely check prefix
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    df = df[~df['InvoiceNo'].str.startswith('C')]

    # 3. Remove outlier prices, non-product entries, and invalid rows
    # Filter out non-product entries often coded as strings in StockCode (e.g., 'POST', 'DOT', 'M', 'BANK CHARGES')
    # Valid stock codes typically consist of digits or standard product SKUs.
    # We will also filter out UnitPrice <= 0.
    df = df[df['UnitPrice'] > 0]

    # Remove rows where StockCode indicates manual fees, postage, or administrative costs
    non_product_codes = ['POST', 'DOT', 'M', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'PADS', 'DCGSSBOY', 'DCGSSGIRL']
    df = df[~df['StockCode'].isin(non_product_codes)]

    # Optional: Remove extreme price outliers (e.g., data entry errors where UnitPrice is unrealistically high)
    # Using 99.9th percentile as a cutoff safeguard
    price_cap = df['UnitPrice'].quantile(0.999)
    df = df[df['UnitPrice'] <= price_cap]

    # 4. Feature Creation: Calculate TotalAmount
    df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

    # Ensure InvoiceDate is parsed as datetime
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

    final_rows = len(df)
    print(f"Data cleaning complete. Retained {final_rows:,} rows. ({initial_rows - final_rows:,} rows removed).")

    return df

if __name__ == "__main__":
    # Example usage (update path to where your dataset is stored)
    file_path = "/content/drive/MyDrive/Data-Analytics-Projects/online_retail.csv"

    try:
        cleaned_df = load_and_clean_data(file_path)

        # Save cleaned dataset for downstream modeling and Power BI
        cleaned_df.to_csv("/content/drive/MyDrive/Data-Analytics-Projects/cleaned_online_retail.csv", index=False)
        print("Cleaned dataset successfully saved to '/content/drive/MyDrive/Data-Analytics-Projects/cleaned_online_retail.csv'.")

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}. Please check your file path.")

Loading dataset...
Initial row count: 541,909
Data cleaning complete. Retained 396,149 rows. (145,760 rows removed).
Cleaned dataset successfully saved to '/content/drive/MyDrive/Data-Analytics-Projects/cleaned_online_retail.csv'.


In [16]:
import pandas as pd
import numpy as np

def compute_rfm_segments(cleaned_file_path: str) -> pd.DataFrame:
    """
    Computes Recency, Frequency, Monetary (RFM) values, scores them using quartiles,
    and assigns behavioral customer segments.
    """
    print("Loading cleaned dataset...")
    df = pd.read_csv(cleaned_file_path)

    # Ensure InvoiceDate is datetime
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

    # Define a snapshot date (1 day after the last transaction in the dataset)
    snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
    print(f"Snapshot date set to: {snapshot_date.date()}")

    print("Aggregating transactional data to customer level (RFM)...")
    # 1. Calculate Recency, Frequency, and Monetary for each CustomerID
    rfm_df = df.groupby('CustomerID').agg({
        'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency: Days since last purchase
        'InvoiceNo': 'nunique',                                  # Frequency: Total unique orders
        'TotalAmount': 'sum'                                     # Monetary: Total spend
    }).reset_index()

    # Rename columns for clarity
    rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

    # Remove customers with zero or negative spend/frequency if any slipped through
    rfm_df = rfm_df[(rfm_df['Monetary'] > 0) & (rfm_df['Frequency'] > 0)]

    print("Assigning RFM Quartile Scores...")
    # 2. Score RFM using quartiles (1 to 4)
    # Note: For Recency, lower is better (more recent), so we invert the labels (4 for lowest recency/recent, 1 for oldest)
    # For Frequency and Monetary, higher is better, so 4 is highest.
    r_labels = range(4, 0, -1)
    f_labels = range(1, 5)
    m_labels = range(1, 5)

    rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], q=4, labels=r_labels, duplicates='drop')
    rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), q=4, labels=f_labels)
    rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], q=4, labels=m_labels)

    # Convert score columns to integers
    rfm_df['R_Score'] = rfm_df['R_Score'].astype(int)
    rfm_df['F_Score'] = rfm_df['F_Score'].astype(int)
    rfm_df['M_Score'] = rfm_df['M_Score'].astype(int)

    # Combine RFM scores into a string or composite score
    rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)
    rfm_df['RFM_Sum'] = rfm_df['R_Score'] + rfm_df['F_Score'] + rfm_df['M_Score']

    print("Mapping segments to customer personas...")
    # 3. Define Segmentation Logic based on RFM scores
    def segment_customer(row):
        r, f, m = row['R_Score'], row['F_Score'], row['M_Score']

        # Champions: Bought recently, buy often, and spend the most
        if r >= 3 and f >= 3 and m >= 3:
            return 'Champions'
        # Loyal Customers: Buy on a regular basis, spend good money
        elif f >= 3 and m >= 2:
            return 'Loyal Customers'
        # Potential Loyalist: Recent customers with average/above average frequency or spend
        elif r >= 3 and f <= 2:
            return 'Potential Loyalists'
        # Recent Customers: Bought very recently, but low frequency/spend so far
        elif r >= 4 and f == 1:
            return 'Recent Customers'
        # At Risk: Purchased often and spent big, but haven't bought in a long time
        elif r <= 2 and f >= 3 and m >= 3:
            return 'At Risk'
        # Cant Lose Them: Used to buy frequently/spend big, but long time since last purchase
        elif r == 1 and f >= 3 and m >= 3:
            return 'Cannot Lose Them'
        # Hibernating / Lost: Low recency, frequency, and monetary scores
        elif r <= 2 and f <= 2:
            return 'Hibernating'
        else:
            return 'Needs Attention'

    rfm_df['Customer_Segment'] = rfm_df.apply(segment_customer, axis=1)

    print("\nSegment Distribution Summary:")
    print(rfm_df['Customer_Segment'].value_counts())

    return rfm_df

if __name__ == "__main__":
    input_path = "/content/drive/MyDrive/Data-Analytics-Projects/cleaned_online_retail.csv"
    output_path = "/content/drive/MyDrive/Data-Analytics-Projects/customer_rfm_segments.csv"

    try:
        rfm_result = compute_rfm_segments(input_path)
        rfm_result.to_csv(output_path, index=False)
        print(f"\nRFM segmentation dataset successfully saved to '{output_path}'.")
    except FileNotFoundError:
        print(f"Error: Could not find '{input_path}'. Make sure Step 2 was run successfully.")

Loading cleaned dataset...
Snapshot date set to: 2011-12-10
Aggregating transactional data to customer level (RFM)...
Assigning RFM Quartile Scores...
Mapping segments to customer personas...

Segment Distribution Summary:
Customer_Segment
Hibernating            1507
Champions              1318
Loyal Customers         758
Potential Loyalists     658
Needs Attention          89
Name: count, dtype: int64

RFM segmentation dataset successfully saved to '/content/drive/MyDrive/Data-Analytics-Projects/customer_rfm_segments.csv'.


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def build_ltv_model(cleaned_file_path: str):
    """
    Splits data into an observation window and prediction window,
    engineers features, trains an LTV regression model, and evaluates it.
    """
    print("Loading cleaned dataset...")
    df = pd.read_csv(cleaned_file_path)
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

    # 1. Define split dates for Observation vs Prediction window
    # Let's find the min and max dates in the dataset
    min_date = df['InvoiceDate'].min()
    max_date = df['InvoiceDate'].max()

    print(f"Dataset date range: {min_date.date()} to {max_date.date()}")

    # Example split: First 9 months for observation, remaining 3 months for prediction target
    # (Adjust based on your dataset span, e.g., 1-year total dataset)
    observation_cutoff = min_date + pd.DateOffset(months=9)
    print(f"Observation Window: {min_date.date()} to {observation_cutoff.date()}")
    print(f"Prediction Window: {observation_cutoff.date()} to {max_date.date()}")

    # Split data
    df_obs = df[df['InvoiceDate'] <= observation_cutoff]
    df_pred = df[df['InvoiceDate'] > observation_cutoff]

    print("Engineering features from observation window...")
    # 2. Calculate RFM/Behavioral features from the observation period
    snapshot_date = observation_cutoff + pd.Timedelta(days=1)

    X_features = df_obs.groupby('CustomerID').agg({
        'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
        'InvoiceNo': 'nunique',                                  # Frequency
        'TotalAmount': ['sum', 'mean', 'max']                    # Total, Avg, Max spend
    })

    # Flatten multi-index columns
    X_features.columns = ['Recency', 'Frequency', 'Monetary_Total', 'Monetary_Mean', 'Monetary_Max']
    X_features.reset_index(inplace=True)

    print("Calculating future spend (Target Variable) from prediction window...")
    # 3. Calculate target variable: Total spend in the prediction window
    y_target = df_pred.groupby('CustomerID')['TotalAmount'].sum().reset_index()
    y_target.columns = ['CustomerID', 'Future_Spend']

    # Merge features and target on CustomerID
    # Inner join or Left join? Left join ensures we include customers who spent 0 in the future period.
    model_df = pd.merge(X_features, y_target, on='CustomerID', how='left')
    model_df['Future_Spend'] = model_df['Future_Spend'].fillna(0) # Customers with no future purchases

    # 4. Prepare matrices for Machine Learning
    feature_cols = ['Recency', 'Frequency', 'Monetary_Total', 'Monetary_Mean', 'Monetary_Max']
    X = model_df[feature_cols]
    y = model_df['Future_Spend']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Training Random Forest Regressor...")
    # 5. Train Model
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)

    # 6. Evaluate Model
    y_pred = rf_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("\n--- Model Performance Metrics ---")
    print(f"Mean Absolute Error (MAE): £{mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): £{rmse:.2f}")
    print(f"R-Squared ($R^2$): {r2:.4f}")

    # Feature Importance
    importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("\nFeature Importances:")
    print(importances)

    # Save predictions back to a dataframe for Power BI / Streamlit
    model_df['Predicted_Future_Spend'] = rf_model.predict(X)

    return model_df, rf_model

if __name__ == "__main__":
    input_path = "/content/drive/MyDrive/Data-Analytics-Projects/cleaned_online_retail.csv"
    output_path = "/content/drive/MyDrive/Data-Analytics-Projects/customer_ltv_predictions.csv"

    try:
        result_df, model = build_ltv_model(input_path)
        result_df.to_csv(output_path, index=False)
        print(f"\nLTV predictions successfully saved to '{output_path}'.")
    except FileNotFoundError:
        print(f"Error: Could not find '{input_path}'. Make sure Step 2 was run successfully.")

Loading cleaned dataset...
Dataset date range: 2010-12-01 to 2011-12-09
Observation Window: 2010-12-01 to 2011-09-01
Prediction Window: 2011-09-01 to 2011-12-09
Engineering features from observation window...
Calculating future spend (Target Variable) from prediction window...
Training Random Forest Regressor...

--- Model Performance Metrics ---
Mean Absolute Error (MAE): £674.00
Root Mean Squared Error (RMSE): £3355.04
R-Squared ($R^2$): -0.2406

Feature Importances:
Monetary_Total    0.710728
Monetary_Max      0.156534
Monetary_Mean     0.112719
Recency           0.010022
Frequency         0.009998
dtype: float64

LTV predictions successfully saved to '/content/drive/MyDrive/Data-Analytics-Projects/customer_ltv_predictions.csv'.
